In [1]:
# allows us to have visibility on our package without installing it in editing mode
import sys;
if ".." not in sys.path: sys.path.append("..")

import itertools
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister, Gate
from qiskit.converters import circuit_to_dag
from monaqa2.qiskit.utils_numpy import kron, ketbra, bra, ket
from monaqa2.qiskit.utils_qiskit import get_unitary
import sympy as sp
from sympy import latex
from IPython.display import display, Markdown


def lx(expr):
    return latex(expr).replace('*', '')


def _qubit_register_info(qc, qubit):
    bitloc = qc.find_bit(qubit)

    if bitloc.registers:
        qreg, reg_idx = bitloc.registers[0]
        return qreg, reg_idx, bitloc.index

    return None, None, bitloc.index


def _is_aux_qubit(qc, qubit):
    qreg, _, _ = _qubit_register_info(qc, qubit)
    return qreg is not None and qreg.name.startswith("aux")


def _initial_quantikz_label(qc, qubit):
    qreg, reg_idx, global_idx = _qubit_register_info(qc, qubit)

    if qreg is not None:
        name = qreg.name

        if name.startswith("aux"):
            return r'\lstick{$\ket{0}$} \qw'

        if len(qreg) == 1:
            return rf'\lstick{{$\ket{{{name}}}$}} \qw'

        return rf'\lstick{{$\ket{{{name}_{{{reg_idx}}}}}$}} \qw'

    return rf'\lstick{{$\ket{{q_{{{global_idx}}}}}$}} \qw'


def _final_quantikz_label(qc, wire_idx, n_q):
    if wire_idx >= n_q:
        return r'\cw'

    qubit = qc.qubits[wire_idx]

    if _is_aux_qubit(qc, qubit):
        return r'\rstick{$\ket{0}$} \qw'

    return r'\qw'


def _display_name(name, rotate_threshold):
    if len(name) >= rotate_threshold:
        return rf'\smash{{\rotatebox[origin=c]{{90}}{{\scriptsize {name}}}}}'
    return name


def _empty_column(n_q, n_c):
    return [r'\qw' for _ in range(n_q)] + [r'\cw' for _ in range(n_c)]


def _invisible_layer_box(content="", text_color="white"):
    """
    Artificial bottom-row cell.

    - draw=white hides the box border
    - fill=white keeps the background white
    - text=black makes layer numbers visible
    - text=white makes spacer cells invisible
    - \\wireoverride{n} suppresses the black horizontal wire segment
    """
    return (
        rf'\gate[style={{draw=white, fill=white, text={text_color}, '
        rf'inner sep=1pt, minimum width=0.6em}}]{{{content}}} '
        rf'\wireoverride{{n}}'
    )


def _empty_layer_cell():
    return _invisible_layer_box("", text_color="white")


def _layer_number_cell(layer_number):
    return _invisible_layer_box(rf'\scriptsize {layer_number}', text_color="black")


def _is_controlled_x_gate(name, q_indices):
    """
    Detect controlled-X-style gates rendered as controls plus X target.

    Handles:
      - CX
      - CCX
      - MCX
      - MCX variants such as MCX_GRAY, MCX_RECURSIVE, MCX_VCHAIN
      - C3X, C4X, etc.
    """
    if len(q_indices) < 2:
        return False

    if name in {"CX", "CCX"}:
        return True

    if name.startswith("MCX"):
        return True

    if name.startswith("C") and name.endswith("X"):
        middle = name[1:-1]
        return middle.isdigit()

    return False


def _controlled_x_controls_and_target(op, q_indices):
    """
    Qiskit controlled-X gates conventionally order qargs as

        controls..., target

    For MCX-like gates, use op.num_ctrl_qubits when available.
    """
    num_ctrls = getattr(op, "num_ctrl_qubits", None)

    if num_ctrls is None:
        name = op.name.upper()

        if name == "CX":
            num_ctrls = 1
        elif name == "CCX":
            num_ctrls = 2
        else:
            num_ctrls = len(q_indices) - 1

    num_ctrls = int(num_ctrls)

    if num_ctrls < 1 or num_ctrls >= len(q_indices):
        num_ctrls = len(q_indices) - 1

    controls = q_indices[:num_ctrls]
    target = q_indices[num_ctrls]

    return controls, target


def _node_wire_span(name, q_indices, c_indices, total_wires):
    """
    Visual Quantikz occupancy.

    This is intentionally visual, not logical. A multi-wire gate on q[0], q[2]
    visually occupies q[1], so it must block that row inside a rendered column.
    """
    if name == "BARRIER":
        return set(range(total_wires))

    if name == "MEASURE":
        q_idx = q_indices[0]
        c_idx = c_indices[0]
        return set(range(min(q_idx, c_idx), max(q_idx, c_idx) + 1))

    if len(q_indices) > 1:
        return set(range(min(q_indices), max(q_indices) + 1))

    if len(q_indices) == 1:
        return {q_indices[0]}

    if c_indices:
        return set(c_indices)

    return set()


def _normalize_labels(labels):
    if labels is None:
        return None

    labels = [str(label) for label in labels]

    if len(labels) == 0:
        return None

    return labels


def _layout_qubit_indices(qc, node, key):
    layout = getattr(node.op, "layout", None)

    if not isinstance(layout, dict) or key not in layout:
        return []

    values = layout[key]

    if values is None:
        return []

    if isinstance(values, (str, bytes)):
        values = [values]
    else:
        try:
            values = list(values)
        except TypeError:
            values = [values]

    q_indices = [qc.find_bit(q).index for q in node.qargs]
    out = []

    for value in values:
        idx = None

        if isinstance(value, (int, np.integer)):
            value = int(value)

            if 0 <= value < len(q_indices):
                idx = q_indices[value]
            elif value in q_indices:
                idx = value
        else:
            try:
                idx = qc.find_bit(value).index
            except Exception:
                idx = None

        if idx in q_indices and idx not in out:
            out.append(idx)

    return out


def _layout_control_indices(qc, node):
    return _layout_qubit_indices(qc, node, "control")


def _is_layout_gateinput_key(key):
    key = str(key)
    return key in {"l", "r", "a", "b", "c", "s", "k", "x", "y"} or key.startswith("in") or key.startswith("out")


def _layout_gateinput_labels(qc, node):
    layout = getattr(node.op, "layout", None)
    out = {}

    if not isinstance(layout, dict):
        return out

    for key in layout:
        key = str(key)

        if not _is_layout_gateinput_key(key):
            continue

        for idx in _layout_qubit_indices(qc, node, key):
            if idx in out:
                out[idx] = rf'{out[idx]},{key}'
            else:
                out[idx] = key

    return out


def _gateinput_label_for_wire(q_idx, q_indices, labels, layout_gateinputs):
    if q_idx in layout_gateinputs:
        return layout_gateinputs[q_idx]

    if labels and q_idx in q_indices:
        port = q_indices.index(q_idx)

        if port < len(labels):
            return labels[port]

    return None


def _append_gateinput(cell, label):
    if label is None:
        return cell

    return cell + rf' \gateinput{{{label}}}'


def _effective_gateinput_labels(q_indices, labels, layout_gateinputs, wires=None):
    wires = q_indices if wires is None else wires
    out = []

    for q_idx in wires:
        label = _gateinput_label_for_wire(q_idx=q_idx, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)

        if label is not None:
            out.append(label)

    return out


def _render_controlled_x_into_column(col, op, q_indices, labels=None, layout_gateinputs=None):
    layout_gateinputs = {} if layout_gateinputs is None else layout_gateinputs
    controls, target = _controlled_x_controls_and_target(op, q_indices)

    for c in controls:
        label = _gateinput_label_for_wire(q_idx=c, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
        col[c] = _append_gateinput(rf'\ctrl{{{target - c}}}', label)

    label = _gateinput_label_for_wire(q_idx=target, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
    col[target] = _append_gateinput(r'\targX{}', label)


def _render_layout_controlled_gate_into_column(col, q_indices, controls, labels, layout_gateinputs, display_name):
    control_set = set(controls)
    targets = [q_idx for q_idx in q_indices if q_idx not in control_set]

    if len(targets) == 0:
        return False

    target_anchor = min(targets)

    for c in controls:
        label = _gateinput_label_for_wire(q_idx=c, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
        col[c] = _append_gateinput(rf'\ctrl{{{target_anchor - c}}}', label)

    if len(targets) == 1:
        target = targets[0]
        label = _gateinput_label_for_wire(q_idx=target, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
        col[target] = _append_gateinput(rf'\gate{{{display_name}}}', label)
        return True

    min_q, max_q = min(targets), max(targets)

    # A multi-wire gate cannot pass through a row that is already rendered as
    # a control. If this happens, keep the controls visible and render the
    # targets as separate one-wire gate boxes.
    if any(min_q < c < max_q for c in controls):
        for target in targets:
            label = _gateinput_label_for_wire(q_idx=target, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
            col[target] = _append_gateinput(rf'\gate{{{display_name}}}', label)

        return True

    span = max_q - min_q + 1
    gateinput_labels = _effective_gateinput_labels(q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs, wires=targets)

    phantom_str = ""

    if gateinput_labels:
        longest_label = max(gateinput_labels, key=len)
        phantom_str = rf'\phantom{{{longest_label}}}'

    gate_content = f"{phantom_str}{display_name}"
    gate_str = rf'\gate[wires={span}]{{{gate_content}}}'

    label = _gateinput_label_for_wire(q_idx=min_q, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
    col[min_q] = _append_gateinput(gate_str, label)

    for idx in range(min_q + 1, max_q + 1):
        label = _gateinput_label_for_wire(q_idx=idx, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)

        if idx in targets and label is not None:
            col[idx] = rf'\gateinput{{{label}}}'
        else:
            col[idx] = ""

    return True


def _render_node_into_column(col, qc, node, n_q, rotate_threshold):
    op = node.op
    name = op.name.upper()

    q_indices = [qc.find_bit(q).index for q in node.qargs]
    c_indices = [qc.find_bit(c).index + n_q for c in node.cargs]

    labels = _normalize_labels(getattr(op, "port_labels", None))
    layout_gateinputs = _layout_gateinput_labels(qc, node)
    display_name = _display_name(name, rotate_threshold)

    # --- Barrier ---
    if name == "BARRIER":
        for i in range(len(col)):
            col[i] = r'\qw' if i < n_q else r'\cw'
        return

    # --- CX / CCX / MCX as controls plus X target ---
    if _is_controlled_x_gate(name, q_indices):
        _render_controlled_x_into_column(col=col, op=op, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
        return

    # --- Standard CZ ---
    if name == "CZ":
        c, t = q_indices[0], q_indices[1]

        c_label = _gateinput_label_for_wire(q_idx=c, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
        t_label = _gateinput_label_for_wire(q_idx=t, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)

        col[c] = _append_gateinput(rf'\ctrl{{{t - c}}}', c_label)
        col[t] = _append_gateinput(r'\control{}', t_label)
        return

    # --- Generic layout-controlled gates ---
    layout_controls = _layout_control_indices(qc, node)

    if layout_controls:
        rendered = _render_layout_controlled_gate_into_column(col=col, q_indices=q_indices, controls=layout_controls, labels=labels, layout_gateinputs=layout_gateinputs, display_name=display_name)

        if rendered:
            return

    # --- Multi-qubit blocks ---
    if len(q_indices) > 1 and name not in ["MEASURE"]:
        min_q, max_q = min(q_indices), max(q_indices)
        span = max_q - min_q + 1

        gateinput_labels = _effective_gateinput_labels(q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)

        phantom_str = ""

        if gateinput_labels:
            longest_label = max(gateinput_labels, key=len)
            phantom_str = rf'\phantom{{{longest_label}}}'

        gate_content = f"{phantom_str}{display_name}"
        gate_str = rf'\gate[wires={span}]{{{gate_content}}}'

        label = _gateinput_label_for_wire(q_idx=min_q, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
        col[min_q] = _append_gateinput(gate_str, label)

        for idx in range(min_q + 1, max_q + 1):
            label = _gateinput_label_for_wire(q_idx=idx, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)

            if idx in q_indices and label is not None:
                col[idx] = rf'\gateinput{{{label}}}'
            else:
                col[idx] = ""

        return

    # --- Standard gates ---
    if name == "MEASURE":
        q_idx, c_idx = q_indices[0], c_indices[0]
        col[q_idx] = rf'\meter{{}} \vqw{{{c_idx - q_idx}}}'
        return

    if len(q_indices) == 1:
        q_idx = q_indices[0]
        label = _gateinput_label_for_wire(q_idx=q_idx, q_indices=q_indices, labels=labels, layout_gateinputs=layout_gateinputs)
        col[q_idx] = _append_gateinput(rf'\gate{{{display_name}}}', label)
        return


def dag_to_quantikz_optimized(qc, rotate_threshold=4, show_layer_numbers=True, layer_number_start=0):
    dag = circuit_to_dag(qc)

    n_q = qc.num_qubits
    n_c = qc.num_clbits
    total_wires = n_q + n_c

    columns = []
    column_layer_numbers = []

    # Initial lstick column.
    columns.append([_initial_quantikz_label(qc, q) for q in qc.qubits] + [r'\cw' for _ in range(n_c)])
    column_layer_numbers.append(None)

    for layer_number, layer in enumerate(dag.layers(), start=layer_number_start):
        layer_dag = layer["graph"]

        # Keep Qiskit's DAG layers. Only split one DAG layer into multiple
        # rendered Quantikz subcolumns if the drawn objects would overlap.
        subcolumns = []
        occupied_by_subcolumn = []

        for node in layer_dag.op_nodes():
            name = node.op.name.upper()

            q_indices = [qc.find_bit(q).index for q in node.qargs]
            c_indices = [qc.find_bit(c).index + n_q for c in node.cargs]

            occupied = _node_wire_span(name=name, q_indices=q_indices, c_indices=c_indices, total_wires=total_wires)

            placed = False

            for col, used in zip(subcolumns, occupied_by_subcolumn):
                if occupied.isdisjoint(used):
                    _render_node_into_column(col=col, qc=qc, node=node, n_q=n_q, rotate_threshold=rotate_threshold)
                    used.update(occupied)
                    placed = True
                    break

            if not placed:
                col = _empty_column(n_q, n_c)
                _render_node_into_column(col=col, qc=qc, node=node, n_q=n_q, rotate_threshold=rotate_threshold)
                subcolumns.append(col)
                occupied_by_subcolumn.append(set(occupied))

        for col in subcolumns:
            if any(cell not in [r'\qw', r'\cw', ""] for cell in col):
                columns.append(col)
                column_layer_numbers.append(layer_number)

    # Final rstick / end-wire column.
    columns.append([_final_quantikz_label(qc, i, n_q) for i in range(total_wires)])
    column_layer_numbers.append(None)

    latex_lines = [r"\begin{quantikz}"]

    for i in range(total_wires):
        row_cells = [columns[j][i] for j in range(len(columns))]
        line = " & ".join(row_cells)

        if i < total_wires - 1 or show_layer_numbers:
            line += r" \\"

        latex_lines.append(line)

    if show_layer_numbers:
        layer_cells = []

        for layer_number in column_layer_numbers:
            if layer_number is None:
                layer_cells.append(_empty_layer_cell())
            else:
                layer_cells.append(_layer_number_cell(layer_number))

        latex_lines.append(" & ".join(layer_cells) + r" \\")

    latex_lines.append(r"\end{quantikz}")

    return "\n".join(latex_lines)

In [3]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
from monaqa2.qiskit.primitives import Ccx, Cry, Ccry, GivensRotation, ControlledGivensRotation


In [8]:
qr_inputs = [QuantumRegister(1, "c_{1}"), QuantumRegister(1, "c_{2}")]
qr_output = QuantumRegister(1, "t")

lst = qr_inputs + [qr_output]
qbt = [q for reg in qr_inputs for q in reg] + qr_output[:]

gate = Ccx()
qc = QuantumCircuit(*lst)
qc.append(gate, qbt)

print(dag_to_quantikz_optimized(qc.decompose()))

\begin{quantikz}
\lstick{$\ket{c_{1}}$} \qw & \gate{T} & \targX{} & \gate{TDG} & \qw & \targX{} & \qw & \ctrl{2} & \qw & \qw & \ctrl{1} & \qw \\
\lstick{$\ket{c_{2}}$} \qw & \gate{T} & \ctrl{-1} & \targX{} & \gate{T} & \ctrl{-1} & \targX{} & \qw & \gate{TDG} & \targX{} & \targX{} & \qw \\
\lstick{$\ket{t}$} \qw & \gate{H} & \gate{T} & \ctrl{-1} & \gate{TDG} & \qw & \ctrl{-1} & \targX{} & \qw & \ctrl{-1} & \gate{H} & \qw \\
\gate[style={draw=white, fill=white, text=white, inner sep=1pt, minimum width=0.6em}]{} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 0} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 1} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 2} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 3} \wire

In [2]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
from monaqa2.qiskit.arithmetic_hybrid import WallaceTreeAdder

n = 2
F = 2
W = F + 1
M = n + n * (n - 1) // 2

h = np.array([0.9, -0.6, 0.4])
J = np.array([[0.0, -0.75, 0.2], [-0.75, 0.0, 0.35], [0.2, 0.35, 0.0]])

adder = WallaceTreeAdder(M, W)

# WallaceTreeAdder layout:
#   in[1], ..., in[M]: M input words of W bits
#   output:            W-bit output word
#   wallace:           (2W - 1)(M - 2) Wallace scratch qubits
#   carries:           W - 1 final ripple-carry qubits
qr_inputs = [QuantumRegister(W, "h_{1}|"), QuantumRegister(W, "h_{2}|"), QuantumRegister(W, "J_{1,2}|")]
qr_output = QuantumRegister(W, "out")
qr_wallace = QuantumRegister((2 * W - 1) * (M - 2), "w")
qr_carries = QuantumRegister(W - 1, "c")

lst = qr_inputs + [qr_output, qr_wallace, qr_carries]
qbt = [q for reg in qr_inputs for q in reg] + qr_output[:] + qr_wallace[:] + qr_carries[:]

qc = QuantumCircuit(*lst)
qc.append(adder, qbt)

print(dag_to_quantikz_optimized(qc.decompose()))

\begin{quantikz}
\lstick{$\ket{h_{1}|_{0}}$} \qw & \qw & \gate[wires=16]{\phantom{a}\smash{\rotatebox[origin=c]{90}{\scriptsize 3TO2 COMPRESSOR}}} \gateinput{a} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \gate[wires=16]{\phantom{a}\smash{\rotatebox[origin=c]{90}{\scriptsize 3TO2 COMPRESSOR}}} \gateinput{a} & \qw & \qw & \qw & \qw & \qw \\
\lstick{$\ket{h_{1}|_{1}}$} \qw & \qw &  & \gate[wires=16]{\phantom{a}\smash{\rotatebox[origin=c]{90}{\scriptsize 3TO2 COMPRESSOR}}} \gateinput{a} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw &  & \gate[wires=16]{\phantom{a}\smash{\rotatebox[origin=c]{90}{\scriptsize 3TO2 COMPRESSOR}}} \gateinput{a} & \qw & \qw & \qw & \qw \\
\lstick{$\ket{h_{1}|_{2}}$} \qw & \ctrl{12} &  &  & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw &  &  & \ctrl{12} & \qw & \qw & \qw \\
\lstick{$\ket{h_{2}|_{0}}$} \qw & \qw & \gateinput{b} &  & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw &

In [5]:
M

3

In [7]:



from functools import reduce

n = 3
h = np.array([0.9, -0.6, 0.4])
J = np.array([[0.0, -0.75, 0.2], [-0.75, 0.0, 0.35], [0.2, 0.35, 0.0]])

qr_z1 = QuantumRegister(n, 'z^a')
qr_z2 = QuantumRegister(n, 'z^b')
qr_u1 = QuantumRegister(n, 'u^a')
qr_u2 = QuantumRegister(n, 'u^b')
qr_v1 = QuantumRegister(n, 'v^a')
qr_v2 = QuantumRegister(n, 'v^b')
qr_a = QuantumRegister(n, 'a')
qr_b = QuantumRegister(n, 'b')
lst = [qr_z1, qr_z2, qr_u1, qr_u2, qr_v1, qr_v2, qr_a, qr_b]
qbt = qr_z1[:] + qr_z2[:] + qr_u1[:] + qr_u2[:] + qr_v1[:] + qr_v2[:] + qr_a[:] + qr_b[:]
qc = QuantumCircuit(*lst)
qc.append(SelectDeltaHamiltonian(n, h, J), qbt)
print(dag_to_quantikz_optimized(qc.decompose()))

\begin{quantikz}
\lstick{$\ket{z^a_{0}}$} \qw & \gate{Z} & \qw & \qw & \qw & \qw & \qw & \ctrl{18} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw \\
\lstick{$\ket{z^a_{1}}$} \qw & \qw & \qw & \ctrl{18} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw \\
\lstick{$\ket{z^a_{2}}$} \qw & \gate{Z} & \qw & \qw & \qw & \qw & \qw & \qw & \ctrl{18} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw \\
\lstick{$\ket{z^b_{0}}$} \qw & \qw & \qw & \qw & \ctrl{18} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw \\
\lstick{$\ket{z^b_{1}}$} \qw & \gate{Z} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \ctrl{18} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw \\
\lstick{$\ket{z^b_{2}}$} \qw & \qw & \qw & \qw & \qw & \ctrl{18} & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw & \qw

Multi controlled not

In [16]:
qc = QuantumCircuit(4)
qc.mcx([0, 1, 2], 3, mode='recursion')
print(qc.decompose().draw())

     ┌────────┐                                                              »
q_0: ┤ P(π/8) ├────■──────────────────■────────────────────■─────────────────»
     ├────────┤  ┌─┴─┐   ┌─────────┐┌─┴─┐                  │                 »
q_1: ┤ P(π/8) ├──┤ X ├───┤ P(-π/8) ├┤ X ├──■───────────────┼──────────────■──»
     ├────────┤  └───┘   └─────────┘└───┘┌─┴─┐┌─────────┐┌─┴─┐┌────────┐┌─┴─┐»
q_2: ┤ P(π/8) ├──────────────────────────┤ X ├┤ P(-π/8) ├┤ X ├┤ P(π/8) ├┤ X ├»
     └─┬───┬──┘┌────────┐                └───┘└─────────┘└───┘└────────┘└───┘»
q_3: ──┤ H ├───┤ P(π/8) ├────────────────────────────────────────────────────»
       └───┘   └────────┘                                                    »
«                                                                         »
«q_0: ─────────────■───────────────────────────────────────────────────■──»
«                  │                                                   │  »
«q_1: ─────────────┼────────────────────■────────────────────

/tmp/ipykernel_7039/3924805308.py:2: DeprecationWarning: ``qiskit.circuit.quantumcircuit.QuantumCircuit.mcx()``'s argument ``mode`` is deprecated as of Qiskit 2.1. It will be removed no earlier than 3 months after the release date. Instead, add a generic MCXGate to the circuit and specify the synthesis method via the ``hls_config`` in the transpilation. Alternatively, specific decompositions are available at https://qisk.it/mcx.
  qc.mcx([0, 1, 2], 3, mode='recursion')


In [8]:
# from monaqa2.qiskit.multi_controlled_not_gate import MultiControlledNot
# controls = 5
# mcx = MultiControlledNot(controls)
# qr_c = QuantumRegister(controls, 'c')
# qr_t = QuantumRegister(1, 't')
# qr_aux = QuantumRegister(mcx.num_qubits - controls - 1, 'aux')
# qc = QuantumCircuit(qr_c, qr_t, qr_aux)
# qc.append(mcx, qr_c[:] + qr_t[:] + qr_aux[:])
# print(dag_to_quantikz_optimized(qc.decompose()))

Reflection

In [9]:
# from monaqa2.qiskit.reflection_gate import Reflection
# from qiskit import QuantumCircuit, transpile, QuantumRegister
# n, c = 3, 1
# refl = Reflection(n=n, coins=c)
# qr_a = QuantumRegister(n, 'a')
# qr_b = QuantumRegister(n, 'b')
# qr_c = QuantumRegister(c, 'coin')
# qr_aux = QuantumRegister(c, 'aux')
# qc = QuantumCircuit(qr_a, qr_b, qr_c, qr_aux)
# qc.append(refl, qr_a[:] + qr_b[:] + qr_c[:] + qr_aux[:])
# print(dag_to_quantikz_optimized(qc.decompose()))

Accept path

In [10]:
# from monaqa2.qiskit.accept_path_gate import AcceptPath
# from qiskit import QuantumCircuit, transpile, QuantumRegister
# n, c = 3, 1
# accp = AcceptPath(n=n, coins=c)
# qr_a = QuantumRegister(n, 'a')
# qr_b = QuantumRegister(n, 'b')
# qr_c = QuantumRegister(c, 'coin')
# qr_aux = QuantumRegister(accp.num_qubits - n - n - c, 'aux')
# qc = QuantumCircuit(qr_a, qr_b, qr_c, qr_aux)
# qc.append(accp, qr_a[:] + qr_b[:] + qr_c[:] + qr_aux[:])
# print(dag_to_quantikz_optimized(qc.decompose()))

Proposal for uniform move

In [11]:
# from monaqa2.qiskit.proposal_uniform_gate import ProposalUniform
# 
# n = 3
# prop_move = ProposalUniform(n=n)
# qr_a = QuantumRegister(n, 'a')
# qr_b = QuantumRegister(n, 'b')
# qc = QuantumCircuit(qr_a, qr_b)
# qc.append(prop_move, qr_a[:] + qr_b[:])
# print(dag_to_quantikz_optimized(qc.decompose()))

Dicke state prep

In [18]:
from monaqa2.qiskit.dicke_preparation_gate import SCS, WDB, DickePreparation
from qiskit import QuantumCircuit, QuantumRegister

n, k = 5, 2

scs = SCS(n=n, k=k)
qr_scs = QuantumRegister(scs.num_qubits, "scs")
qc_scs = QuantumCircuit(qr_scs)
qc_scs.append(scs, qr_scs[:])
print("SCS block")
print(dag_to_quantikz_optimized(qc_scs.decompose()))

SCS block
\begin{quantikz}
\lstick{$\ket{scs_{0}}$} \qw & \qw & \qw & \qw & \ctrl{2} & \gate{RY} & \targX{} & \gate{RY} & \targX{} & \ctrl{2} & \qw \\
\lstick{$\ket{scs_{1}}$} \qw & \ctrl{1} & \gate[wires=2]{CRY} & \ctrl{1} & \qw & \qw & \ctrl{-1} & \qw & \ctrl{-1} & \qw & \qw \\
\lstick{$\ket{scs_{2}}$} \qw & \targX{} &  & \targX{} & \targX{} & \qw & \ctrl{-2} & \qw & \ctrl{-2} & \targX{} & \qw \\
\gate[style={draw=white, fill=white, text=white, inner sep=1pt, minimum width=0.6em}]{} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 0} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 1} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 2} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 3} \wireoverride{n} & \gate[style

In [19]:
wdb = WDB(n=n, m=1, k=k)
qr_wdb = QuantumRegister(wdb.num_qubits, "wdb")
qc_wdb = QuantumCircuit(qr_wdb)
qc_wdb.append(wdb, qr_wdb[:])
print("WDB block")
print(dag_to_quantikz_optimized(qc_wdb.decompose()))

WDB block
\begin{quantikz}
\lstick{$\ket{wdb_{0}}$} \qw & \targX{} & \qw & \gate[wires=5]{CRY} & \targX{} & \gate[wires=5]{\smash{\rotatebox[origin=c]{90}{\scriptsize CSWAP}}} & \qw & \qw \\
\lstick{$\ket{wdb_{1}}$} \qw & \ctrl{-1} & \gate[wires=4]{CRY} &  & \ctrl{-1} &  & \targX{} & \qw \\
\lstick{$\ket{wdb_{2}}$} \qw & \qw &  &  & \qw &  & \qw & \qw \\
\lstick{$\ket{wdb_{3}}$} \qw & \qw &  &  & \qw &  & \qw & \qw \\
\lstick{$\ket{wdb_{4}}$} \qw & \qw &  &  & \qw &  & \ctrl{-3} & \qw \\
\gate[style={draw=white, fill=white, text=white, inner sep=1pt, minimum width=0.6em}]{} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 0} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 1} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 2} \wireoverride{n} & \gate[style={draw=white, fill=white, tex

In [20]:
dicke = DickePreparation(n=n, k=k)
qr_dicke = QuantumRegister(dicke.num_qubits, "dicke")
qc_dicke = QuantumCircuit(qr_dicke)
qc_dicke.append(dicke, qr_dicke[:])
print("Dicke state preparation")
print(dag_to_quantikz_optimized(qc_dicke.decompose()))

Dicke state preparation
\begin{quantikz}
\lstick{$\ket{dicke_{0}}$} \qw & \gate{X} & \gate[wires=5]{WDB} & \gate[wires=2]{\smash{\rotatebox[origin=c]{90}{\scriptsize SWAP}}} & \gate[wires=2]{SCS} & \qw & \qw \\
\lstick{$\ket{dicke_{1}}$} \qw & \gate{X} &  &  &  & \qw & \qw \\
\lstick{$\ket{dicke_{2}}$} \qw & \qw &  & \gate[wires=3]{WDB} & \gate[wires=2]{\smash{\rotatebox[origin=c]{90}{\scriptsize SWAP}}} & \gate[wires=2]{SCS} & \qw \\
\lstick{$\ket{dicke_{3}}$} \qw & \qw &  &  &  &  & \qw \\
\lstick{$\ket{dicke_{4}}$} \qw & \qw &  &  & \qw & \qw & \qw \\
\gate[style={draw=white, fill=white, text=white, inner sep=1pt, minimum width=0.6em}]{} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 0} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scriptsize 1} \wireoverride{n} & \gate[style={draw=white, fill=white, text=black, inner sep=1pt, minimum width=0.6em}]{\scrip